# matmul-back-transpose-pair — worked example 2: Backward Pass Through a Manual Linear Layer

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `matmul-back-transpose-pair`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A linear layer computes `out = x @ W.T + b` where `x: (B, in_dim)`, `W: (out_dim, in_dim)`. The weight gradient is `grad_out.T @ x` and the input gradient is `grad_out @ W`. The bias gradient is `grad_out.sum(dim=0)`. This is the complete backward pass for `nn.Linear` implemented manually using the transpose-pair rule.

## Worked solution

**Step 1 — forward pass.**
Given `x: (B, in_dim)`, `W: (out_dim, in_dim)`, `b: (out_dim,)`, compute `out = x @ W.T + b`. The shape is `(B, in_dim) @ (in_dim, out_dim) → (B, out_dim)` plus broadcast bias.

**Step 2 — gradient w.r.t. W.**
We need `dL/dW` of shape `(out_dim, in_dim)`. The forward was `out = x @ W.T`, so thinking of it as `out = x @ (W.T)`, the gradient w.r.t. `W.T` is `x.T @ grad_out: (in_dim, B) @ (B, out_dim) = (in_dim, out_dim)`. Then `dL/dW = (x.T @ grad_out).T = grad_out.T @ x`.

**Step 3 — gradient w.r.t. x.**
We need `dL/dx` of shape `(B, in_dim)`. The forward was `x @ W.T`, so `dL/dx = grad_out @ W: (B, out_dim) @ (out_dim, in_dim) = (B, in_dim)`.

**Step 4 — gradient w.r.t. b.**
The bias was broadcast over the batch dimension, so we sum `grad_out` over the batch: `dL/db = grad_out.sum(dim=0): (out_dim,)`.

In [ ]:
import torch as t

t.manual_seed(33)
B, in_dim, out_dim = 6, 4, 3

x = t.randn(B, in_dim)
W = t.randn(out_dim, in_dim)
b = t.randn(out_dim)

# Forward: out = x @ W.T + b
out = x @ W.T + b   # (B, out_dim)
grad_out = t.randn_like(out)

# Manual backward using the transpose-pair rule
dL_dW = grad_out.T @ x          # (out_dim, B) @ (B, in_dim) = (out_dim, in_dim)
dL_dx = grad_out @ W            # (B, out_dim) @ (out_dim, in_dim) = (B, in_dim)
dL_db = grad_out.sum(dim=0)     # (out_dim,)

print(f"x: {x.shape}, W: {W.shape}, b: {b.shape}")
print(f"out: {out.shape}")
print(f"dL/dW shape: {dL_dW.shape}  expected ({out_dim},{in_dim})")
print(f"dL/dx shape: {dL_dx.shape}  expected ({B},{in_dim})")
print(f"dL/db shape: {dL_db.shape}  expected ({out_dim},)")

# Verify with autograd
x_ag = x.clone().requires_grad_(True)
W_ag = W.clone().requires_grad_(True)
b_ag = b.clone().requires_grad_(True)
out_ag = x_ag @ W_ag.T + b_ag
out_ag.backward(grad_out)

print(f"\ndL/dW match: {t.allclose(dL_dW, W_ag.grad, atol=1e-5)}")
print(f"dL/dx match: {t.allclose(dL_dx, x_ag.grad, atol=1e-5)}")
print(f"dL/db match: {t.allclose(dL_db, b_ag.grad, atol=1e-5)}")